# Rootbeer — Transactions pipeline (Alteryx → PySpark)
Mirrors **Transactions – Transform Data** in `Rootbeer Alteryx Project.yxmd`: input hygiene, time fields, days-since-last-purchase (with imputation), joins to `rootbeer.csv` and `geolocation.csv`, optional brand URLs, **Time on Shelf**.

**Prerequisite:** **`notebook_00_setup_configuration`** is `%run` below. CSVs are read from **`source_data_path`**. This notebook **persists** DataFrames to the **bronze / silver / gold** (and **fact / dim**) table names defined in `pipeline_config`.

In [0]:
%run ./notebook_00_setup_configuration

# 00 — Setup and configuration (Alteryx → Databricks)

**Purpose:** Unity Catalog names, ADLS-backed storage paths, Spark settings, and a shared `pipeline_config` temp view for downstream notebooks (`01` bronze, `02` silver, etc.).

**Naming convention**
- **Catalog** — top-level UC boundary (e.g. training workspace catalog).
- **Schemas** — `{user}_{layer}_alteryx_rootbeer`: bronze = raw Designer exports, silver = cleansed joins / macro parity, gold = KPIs and reporting tables.
- **Tables** — `{layer}_alteryx_rootbeer_{entity}` (e.g. `bronze_alteryx_rootbeer_transaction`). **Fact** and **dimension** tables from config are written in **`SILVER_SCHEMA`** in notebook 01 (star-schema naming); **gold** tables hold aggregates.
- **Paths** — under an ADLS container exposed as a **Unity Catalog external volume** or **`abfss://`** URI. Adjust `STORAGE_ACCOUNT` and `ADLS_CONTAINER` to match your landing zone.

## Configuration parameters

✅ Configuration loaded
   Catalog: `na-dbxtraining`
   Bronze schema: biju_bronze
   Silver schema: biju_silver
   Gold schema: biju_gold
   Source (Volume): /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/source/
   ADLS base (abfss): abfss://landingzone-biju@dlsdbxtraining002.dfs.core.windows.net/alteryx


## Create schemas

✅ Schemas ensured:
   Bronze: ``na-dbxtraining``.biju_bronze
   Silver: ``na-dbxtraining``.biju_silver
   Gold:   ``na-dbxtraining``.biju_gold


## Spark configuration

## Save configuration for other notebooks

✅ Configuration saved to temp view `pipeline_config`

📋 Full configuration:
   adls_base_path: abfss://landingzone-biju@dlsdbxtraining002.dfs.core.windows.net/alteryx
   adls_container: landingzone-biju
   broadcast_threshold: 10485760
   bronze_checkpoint_path: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/checkpoints/bronze/
   bronze_schema: biju_bronze
   bronze_table_customer: bronze_alteryx_rootbeer_customer
   bronze_table_geolocation: bronze_alteryx_rootbeer_geolocation
   bronze_table_google_trends: bronze_alteryx_rootbeer_google_trends
   bronze_table_rootbeer: bronze_alteryx_rootbeer_product
   bronze_table_rootbeer_review: bronze_alteryx_rootbeer_review
   bronze_table_transaction: bronze_alteryx_rootbeer_transaction
   catalog: `na-dbxtraining`
   checkpoint_location: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/checkpoints/bronze/
   cluster_columns_csv: transaction_date,creditcard_type,location_id
   dim_creditcard_type: dim_alteryx_rootbeer_cr

## Ready

- **`notebook_01_rootbeer_transactions_pipeline.py`** and **`notebook_02_google_trends_union.py`** call `%run ./notebook_00_setup_configuration` and read paths from **`pipeline_config`** (`source_data_path`, etc.). You can still run this notebook alone to refresh schemas, Spark settings, and the temp view.
- **ADLS:** External location / volume should cover `abfss://alteryx@dlsdbxtraining002.dfs.core.windows.net/` (container `alteryx`, account `dlsdbxtraining002`). Subfolder `rootbeer/` matches `ADLS_BASE_PATH`.

In [0]:
%run ./transform

# transform — Generic DataFrame Transformation Utilities
Shared helper library for all Alteryx-converted notebooks.
Use via `%run ./transform` at the top of any notebook.

[transform] Utilities loaded: read_csv, write_output, save_as_delta_table,
            save_as_uc_table_from_config, filter_rows, drop_nulls, drop_duplicates,
            join_dfs, add_column, rename_cols, cast_cols, select_cols, summarize, preview
[transform] Rootbeer macros: state_names_dataframe, macro_imputation_v3_fill_null,
            macro_multi_field_formula_brand_urls, macro_cleanse_review_text,
            macro_batch_summarize_revenue_profit, batch_macro_creditcard_revenue_profit_from_volume,
            add_placeholder_profit, standard_macro_maven_join,
            build_rootbeer_enriched_transactions, read_google_rootbeer_union


In [0]:
_rows = spark.sql("SELECT * FROM pipeline_config LIMIT 1").collect()
if not _rows:
    raise RuntimeError("pipeline_config is empty; run notebook_00_setup_configuration first.")
cfg = _rows[0].asDict()
BASE_PATH = cfg["source_data_path"].rstrip("/")
print(f"Using source_data_path: {BASE_PATH}")

Using source_data_path: /Volumes/na-dbxtraining/biju_raw/biju_vol/alteryx/rootbeer/source


## Bronze — raw CSV → Delta (config table names)

In [0]:
import os
from pyspark.sql import functions as F
from pyspark.sql import Window

cfg["catalog"] = cfg["catalog"].strip("`")

_bronze_files = [
    ("transaction.csv", "bronze_table_transaction"),
    ("customer.csv", "bronze_table_customer"),
    ("rootbeer.csv", "bronze_table_rootbeer"),
    ("rootbeerreview.csv", "bronze_table_rootbeer_review"),
    ("geolocation.csv", "bronze_table_geolocation"),
]

for fname, table_key in _bronze_files:
    fp = f"{BASE_PATH}/{fname}"
    if not os.path.exists(fp):
        print(f"[bronze] skip (file missing): {fp}")
        continue
    _df = read_csv(fp)
    save_as_uc_table_from_config(_df, cfg, schema_key="bronze_schema", table_key=table_key)

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_transaction`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_customer`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_product`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_review`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_bronze`.`bronze_alteryx_rootbeer_geolocation`


## Silver — conformed copies (`silver_table_customer`, `silver_table_review`, optional brand)

In [0]:
_scust = f"{BASE_PATH}/customer.csv"
if os.path.exists(_scust):
    _sdf = read_csv(_scust)
    if "customerid" in _sdf.columns:
        _sdf = _sdf.withColumn("customerid", F.col("customerid").cast("int"))
    if "zipcode" in _sdf.columns:
        _sdf = _sdf.withColumn("zipcode", F.col("zipcode").cast("long"))
    save_as_uc_table_from_config(_sdf, cfg, schema_key="silver_schema", table_key="silver_table_customer")

_srev = f"{BASE_PATH}/rootbeerreview.csv"
if os.path.exists(_srev):
    _srv = read_csv(_srev)
    for c, t in [("customerid", "int"), ("brandid", "int"), ("starrating", "int")]:
        if c in _srv.columns:
            _srv = _srv.withColumn(c, F.col(c).cast(t))
    if "reviewdate" in _srv.columns:
        _srv = _srv.withColumn("reviewdate", F.to_date(F.col("reviewdate")))
    save_as_uc_table_from_config(_srv, cfg, schema_key="silver_schema", table_key="silver_table_review")

_sbrand = f"{BASE_PATH}/rootbeerbrand.csv"
if os.path.exists(_sbrand):
    _sbr = read_csv(_sbrand)
    save_as_uc_table_from_config(_sbr, cfg, schema_key="silver_schema", table_key="silver_table_brand")
else:
    print("[silver] skip silver_table_brand (rootbeerbrand.csv not in source path — export from Designer if needed)")

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`silver_alteryx_rootbeer_customer`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`silver_alteryx_rootbeer_review`
[silver] skip silver_table_brand (rootbeerbrand.csv not in source path — export from Designer if needed)


## Silver — enriched transactions (Alteryx workflow parity)

In [0]:
enriched_tx = build_rootbeer_enriched_transactions(
    BASE_PATH,
    include_brand_join=False,
    brand_csv_path=None,
)

_enriched_clean = enriched_tx
for c in _enriched_clean.columns:
    _enriched_clean = _enriched_clean.withColumnRenamed(c, c.replace(" ", "_"))

save_as_uc_table_from_config(
    _enriched_clean,
    cfg,
    schema_key="silver_schema",
    table_key="silver_table_transaction_enriched",
)

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`silver_alteryx_rootbeer_transaction_enriched`


In [0]:
preview(enriched_tx, 15, label="enriched transactions")

## Silver — fact & dimensions (star schema names from config)

In [0]:
# Fact = same grain as enriched transactions (one row per transaction).
_fact_tx = enriched_tx
for c in _fact_tx.columns:
    _fact_tx = _fact_tx.withColumnRenamed(c, c.replace(" ", "_"))
save_as_uc_table_from_config(
    _fact_tx,
    cfg,
    schema_key="silver_schema",
    table_key="fact_table",
)

# dim_location ← geolocation.csv
_geo_path = f"{BASE_PATH}/geolocation.csv"
if os.path.exists(_geo_path):
    _geo = read_csv(_geo_path)
    save_as_uc_table_from_config(_geo, cfg, schema_key="silver_schema", table_key="dim_location")

# dim_customer ← customer.csv (one row per customerid)
_cust_path = f"{BASE_PATH}/customer.csv"
if os.path.exists(_cust_path):
    _cust = read_csv(_cust_path)
    if "customerid" in _cust.columns:
        _cust = _cust.dropDuplicates(["customerid"])
    save_as_uc_table_from_config(_cust, cfg, schema_key="silver_schema", table_key="dim_customer")

# dim_rootbeer_product ← rootbeer.csv
_rb_path = f"{BASE_PATH}/rootbeer.csv"
if os.path.exists(_rb_path):
    _rb = read_csv(_rb_path)
    if "rootbeerid" in _rb.columns:
        _rb = _rb.dropDuplicates(["rootbeerid"])
    save_as_uc_table_from_config(_rb, cfg, schema_key="silver_schema", table_key="dim_rootbeer_product")

# dim_creditcard_type ← distinct card types from raw transactions
_tx_path = f"{BASE_PATH}/transaction.csv"
if os.path.exists(_tx_path):
    _txr = read_csv(_tx_path)
    if "creditcardtype" in _txr.columns:
        _ccd = (
            _txr.select(F.col("creditcardtype").alias("creditcard_type"))
            .where(F.col("creditcard_type").isNotNull())
            .distinct()
        )
        _w = Window.orderBy("creditcard_type")
        _ccd = _ccd.withColumn("creditcard_type_id", F.row_number().over(_w))
        save_as_uc_table_from_config(_ccd, cfg, schema_key="silver_schema", table_key="dim_creditcard_type")

# dim_date ← distinct transaction dates from enriched stream
_dimd = (
    enriched_tx.select(F.col("Transaction Date").alias("date_key"))
    .where(F.col("date_key").isNotNull())
    .distinct()
)
_dimd = _dimd.select(
    "date_key",
    F.year("date_key").alias("cal_year"),
    F.month("date_key").alias("cal_month"),
    F.dayofmonth("date_key").alias("cal_day"),
    F.date_format("date_key", "EEEE").alias("day_name"),
)
save_as_uc_table_from_config(_dimd, cfg, schema_key="silver_schema", table_key="dim_date")

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`fact_alteryx_rootbeer_transaction`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`dim_alteryx_rootbeer_location`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`dim_alteryx_rootbeer_customer`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`dim_alteryx_rootbeer_product`


/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/expressions.py:968: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`dim_alteryx_rootbeer_creditcard_type`
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_silver`.`dim_alteryx_rootbeer_date`


## Gold — batch macro & daily summary

In [0]:
# Batch macro (Tool 226) — raw `transaction.csv` + placeholder Profit
batch_summary = batch_macro_creditcard_revenue_profit_from_volume(BASE_PATH)
save_as_uc_table_from_config(
    batch_summary,
    cfg,
    schema_key="gold_schema",
    table_key="gold_creditcard_revenue_summary",
)
preview(batch_summary, 20, label="batch macro: Revenue & Sum_Profit by card type")

# Daily aggregates from enriched fact stream
_gold_daily = enriched_tx.groupBy(F.col("Transaction Date").alias("transaction_date")).agg(
    F.count(F.lit(1)).alias("transaction_count"),
    F.sum(F.col("Purchase Price").cast("double")).alias("total_purchase_price"),
)
save_as_uc_table_from_config(
    _gold_daily,
    cfg,
    schema_key="gold_schema",
    table_key="gold_daily_transaction_summary",
)

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_gold`.`gold_alteryx_rootbeer_creditcard_revenue_profit`
[batch macro: Revenue & Sum_Profit by card type] Schema:
root
 |-- creditcardtype: string (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Sum_Profit: double (nullable = true)

[batch macro: Revenue & Sum_Profit by card type] Sample (20 rows):
+----------------+-------+----------+
|creditcardtype  |Revenue|Sum_Profit|
+----------------+-------+----------+
|Discover        |3256.0 |2148.0    |
|MasterCard      |6107.0 |4020.0    |
|American Express|2829.0 |1864.0    |
|Visa            |6304.0 |4152.0    |
+----------------+-------+----------+

[batch macro: Revenue & Sum_Profit by card type] Total rows: 4
[save_as_uc_table_from_config] `na-dbxtraining`.`biju_gold`.`gold_alteryx_rootbeer_daily_transaction_summary`


## Gold — standard macro (customer × reviews × state names)

In [0]:
# Standard macro (Tool 225) — customers + reviews + US state names (State Names.xlsx parity)
customers = read_csv(f"{BASE_PATH}/customer.csv")
reviews = read_csv(f"{BASE_PATH}/rootbeerreview.csv")
for c, t in [("customerid", "int"), ("brandid", "int")]:
    if c in reviews.columns:
        reviews = reviews.withColumn(c, F.col(c).cast(t))
if "customerid" in customers.columns:
    customers = customers.withColumn("customerid", F.col("customerid").cast("int"))

maven_preview = standard_macro_maven_join(customers, reviews, spark=spark)
save_as_uc_table_from_config(
    maven_preview,
    cfg,
    schema_key="gold_schema",
    table_key="gold_customer_review_engagement",
)
preview(maven_preview, 15, label="standard macro (approx)")

[save_as_uc_table_from_config] `na-dbxtraining`.`biju_gold`.`gold_alteryx_rootbeer_customer_review_engagement`
[standard macro (approx)] Schema:
root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- fullname: string (nullable = true)
 |-- streetaddress: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- phonenumber: string (nullable = true)
 |-- firstpurchasedate: date (nullable = true)
 |-- subscribedtoemaillist: boolean (nullable = true)
 |-- gender: string (nullable = true)
 |-- brandid: integer (nullable = true)
 |-- starrating: integer (nullable = true)
 |-- reviewdate: date (nullable = true)
 |-- review: string (nullable = true)

[standard macro (approx)] Sample (15 rows):
+----------+---------+---------+------------------+----------------------+----------+----------+-------

DataFrame[customerid: int, firstname: string, lastname: string, fullname: string, streetaddress: string, city: string, state: string, zipcode: int, email: string, phonenumber: string, firstpurchasedate: date, subscribedtoemaillist: boolean, gender: string, brandid: int, starrating: int, reviewdate: date, review: string]

## Optional file export

Uncomment to also write Parquet under **`gold_output_path`** from config.

In [0]:
# write_output(enriched_tx, f"{cfg.get('gold_output_path', BASE_PATH).rstrip('/')}/enriched_transactions", fmt="parquet")